# Fine-Tuning PIC (Pluralistic Image Completion) on Flowers

Fine-tunes the **PIC** inpainting model — currently used with its downloaded **ImageNet**
weights — on our Oxford-102 **flowers** dataset, and compares *pretrained vs. fine-tuned*
completions.

**Objective:** VAE-only (KL + multi-scale L1, no discriminators) — a faithful port of the
non-adversarial terms of PIC's original training loop, which is stable for a short fine-tune.

**This notebook:**
1. Maintains a **validation-loss curve, updated every epoch**.
2. **Exports weights each epoch** and keeps **`best_*`** weights (lowest validation hole-L1).
3. **Runs inference on a fixed validation image every few epochs** and displays the result.
4. Ends with a **pretrained-vs-fine-tuned comparison** (grid + metric table).

## How to run (Google Colab)
1. Runtime → Change runtime type → **GPU**.
2. Upload **`pic_finetune_bundle.zip`** (from `build_bundle.py`) using the Files pane, or run the
   upload cell below.
3. Run all cells top to bottom. Fine-tuned weights land in `checkpoints/finetuned/`.

## 1. Setup — unzip the bundle and set up import paths

In [ ]:
# In Colab, upload the bundle first if it isn't already here:
#   from google.colab import files; files.upload()   # pick pic_finetune_bundle.zip
import os, sys, glob, zipfile

# Locate the bundle (exact name, or any pic_finetune_bundle*.zip Colab may have renamed).
BUNDLE = "pic_finetune_bundle.zip"
if not os.path.exists(BUNDLE):
    hits = sorted(glob.glob("pic_finetune_bundle*.zip")) or sorted(glob.glob("*.zip"))
    if hits:
        BUNDLE = hits[0]

# Extract with Python's zipfile (always available; raises real errors, unlike `!unzip`).
if not os.path.isdir("pic_repo"):
    if not os.path.exists(BUNDLE):
        raise FileNotFoundError(
            "Bundle not found. Upload pic_finetune_bundle.zip to this working directory "
            "(Colab: Files pane, or run `from google.colab import files; files.upload()`), "
            "then re-run this cell. Files here: %s" % os.listdir(".")
        )
    print("extracting", BUNDLE, "...")
    with zipfile.ZipFile(BUNDLE) as zf:
        zf.extractall(".")
    print("extracted.")
else:
    print("already extracted (pic_repo/ present).")

# The bundle unpacks a flat layout: shared_utils.py, pic_inference.py,
# pic_finetune_core.py, pic_repo/, checkpoints/pretrained/, data_128x128/.
ROOT = os.getcwd()
for p in [ROOT, os.path.join(ROOT, "pic_repo")]:
    if p not in sys.path:
        sys.path.insert(0, p)

assert os.path.isdir("data_128x128"), "data_128x128/ missing after extract"
assert os.path.isdir("checkpoints/pretrained"), "checkpoints/pretrained/ missing after extract"
print("data images found:", len(glob.glob("data_128x128/*.jpg")))
print("pretrained weights:", sorted(os.listdir("checkpoints/pretrained")))

## 2. Configuration

In [ ]:
import torch

EPOCHS_TO_RUN = 30        # how many epochs to train THIS run (re-run the loop cell to add more)
BATCH_SIZE   = 8
LR           = 1e-4       # generator + encoder LR
BETAS        = (0.0, 0.999)
LAMBDA_KL    = 20.0
LAMBDA_REC   = 20.0
OUTPUT_SCALE = 4          # must match the pretrained architecture (do not change)
VAL_FRACTION = 0.1
VIS_EVERY    = 5          # run + show inference on the fixed val image every N epochs
SAMPLE_NUM   = 3          # diverse completions to draw in visualizations
NUM_WORKERS  = 2
SEED         = 42

PRETRAINED_DIR = "checkpoints/pretrained"
OUTPUT_DIR     = "checkpoints/finetuned"
EVAL_DIR       = "evaluations/pic_finetuning"

# --- resume ---
# RESUME=True picks up from the highest epoch_N_net_{E,G}.pth in RESUME_DIR and continues
# from epoch N+1. If a train_state.pt sits alongside them, the optimizer + loss history are
# restored too (faithful resume); otherwise it resumes from the weights with a fresh optimizer.
RESUME     = False
RESUME_DIR = OUTPUT_DIR   # defaults to OUTPUT_DIR (its own saved epochs)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}   output: {OUTPUT_DIR}")

## 3. Imports & data

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

from shared_utils import ImageDataset, get_large_random_mask
from pic_inference import get_pic_inpainter, pic_inpaint
import pic_finetune_core as core

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Flowers loaded as [0,1] 128x128 tensors (Resize is a no-op for data_128x128).
transform = transforms.Compose([transforms.Resize((128, 128)), transforms.ToTensor()])
dataset = ImageDataset(directory="data_128x128", transform=transform)

n_val = max(1, int(len(dataset) * VAL_FRACTION))
n_train = len(dataset) - n_val
train_set, val_set = random_split(
    dataset, [n_train, n_val], generator=torch.Generator().manual_seed(SEED)
)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, drop_last=True)
print(f"train: {n_train}   val: {n_val}")

### Fix the validation set, its masks, and one preview image

The validation masks are generated **once** (seeded) so the per-epoch validation curve is
comparable across epochs. We also pin a single image + mask for the periodic preview.

In [ ]:
# Materialize the whole (small) validation set once, plus fixed 1=hole masks.
val_images = torch.stack([val_set[i] for i in range(len(val_set))])          # (Nval,3,128,128)
g = torch.Generator().manual_seed(SEED)
_saved = torch.get_rng_state()
torch.manual_seed(SEED)                    # get_large_random_mask uses python `random`
random.seed(SEED)
val_masks = get_large_random_mask(val_images.size(0), 128, 128, torch.device("cpu"))  # 1=hole
torch.set_rng_state(_saved)

# One fixed preview sample (first val image + its fixed mask).
vis_image = val_images[:1]
vis_mask  = val_masks[:1]
print("val_images:", tuple(val_images.shape), " val_masks:", tuple(val_masks.shape))

## 4. Build the model from the pretrained ImageNet weights

In [ ]:
from itertools import chain

# Always build on the pretrained architecture; resume overwrites the weights below.
net_E, net_G = get_pic_inpainter(PRETRAINED_DIR, DEVICE)
optimizer = torch.optim.Adam(
    chain(filter(lambda p: p.requires_grad, net_G.parameters()),
          filter(lambda p: p.requires_grad, net_E.parameters())),
    lr=LR, betas=BETAS,
)

# --- resume from the last saved epoch, if requested ---
start_epoch = 0
resume_state = None
if RESUME:
    last, e_path, g_path = core.find_last_epoch(RESUME_DIR)
    if last > 0:
        core.load_weights(net_E, net_G, e_path, g_path, DEVICE)
        start_epoch = last
        print(f"RESUME: loaded epoch {last} weights from {RESUME_DIR}")
        ts = os.path.join(RESUME_DIR, "train_state.pt")
        if os.path.exists(ts):
            resume_state = core.load_train_state(ts, optimizer, DEVICE)
            start_epoch = resume_state.get("completed_epochs", last)
            print(f"  restored optimizer + history (completed_epochs={start_epoch})")
        else:
            print("  no train_state.pt found -> resuming weights only, fresh optimizer/history")
    else:
        print(f"RESUME on but no epoch_*_net_*.pth in {RESUME_DIR}; starting fresh from pretrained")

net_E.train(); net_G.train()
n_params = sum(p.numel() for p in net_E.parameters()) + sum(p.numel() for p in net_G.parameters())
print(f"encoder+generator params: {n_params/1e6:.2f}M   start_epoch={start_epoch}")

## 5. Visualization helper

Renders `Real | Masked | Sample 1..K` for one or more images, using the shared composition
rule `comp = real*(1-mask) + fill*mask`.

In [ ]:
@torch.no_grad()
def show_completions(net_E, net_G, images, masks, sample_num, title):
    was_training = net_E.training
    net_E.eval(); net_G.eval()
    fills = pic_inpaint(net_E, net_G, images, masks, sample_num=sample_num, device=DEVICE)  # (B,K,3,128,128)
    if was_training:
        net_E.train(); net_G.train()

    images = images.cpu(); masks = masks.cpu(); fills = fills.cpu()
    masked = images * (1.0 - masks)
    B, K = images.size(0), sample_num
    ncol = 2 + K
    fig, axes = plt.subplots(B, ncol, figsize=(2.1 * ncol, 2.1 * B))
    axes = np.array(axes).reshape(B, ncol)
    labels = ["Real", "Masked"] + [f"Sample {k+1}" for k in range(K)]
    for b in range(B):
        comps = [images[b], masked[b]] + [
            (masked[b] + fills[b, k] * masks[b]).clamp(0, 1) for k in range(K)
        ]
        for c in range(ncol):
            axes[b, c].imshow(comps[c].permute(1, 2, 0).numpy())
            axes[b, c].axis("off")
            if b == 0:
                axes[b, c].set_title(labels[c], fontsize=10)
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

## 6. Baseline preview (pretrained, before any fine-tuning)

The same fixed val image we'll track throughout training.

In [ ]:
show_completions(net_E, net_G, vis_image, vis_mask, SAMPLE_NUM,
                 title="Pretrained (epoch 0) — fixed validation image")

## 7. Initialize training state (run once)

Sets up the epoch counter and loss history. If resuming with a `train_state.pt`, they are
restored so the loss curve continues seamlessly; a weights-only resume continues the epoch
numbering from `start_epoch` with empty history.

In [ ]:
import shutil

if resume_state is not None:
    completed_epochs = resume_state["completed_epochs"]
    h = resume_state["history"]
    epoch_hist      = list(h["epoch"])
    train_loss_hist = list(h["train"])
    val_l1_hist     = list(h["val_l1"])
    val_psnr_hist   = list(h["val_psnr"])
    best_val_l1     = resume_state["best_val_l1"]
    best_epoch      = resume_state["best_epoch"]
else:
    completed_epochs = start_epoch            # 0 fresh, or N for a weights-only resume
    epoch_hist, train_loss_hist, val_l1_hist, val_psnr_hist = [], [], [], []
    best_val_l1, best_epoch = float("inf"), -1

print(f"training state ready: completed_epochs={completed_epochs}, "
      f"history points={len(epoch_hist)}, best_val_l1={best_val_l1}")

## 8. Training loop — trains `EPOCHS_TO_RUN` more epochs

**Re-run this cell** to train further: it continues from `completed_epochs` (no reset). Each
epoch trains on the flowers, computes validation hole-L1 / PSNR, redraws the loss curve,
checkpoints (`epoch_N` + `latest_*` + `best_*` + `train_state.pt`), and every `VIS_EVERY`
epochs shows the fixed preview.

In [ ]:
def plot_curves():
    clear_output(wait=True)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(epoch_hist, train_loss_hist, "-o", label="train total loss")
    ax[0].plot(epoch_hist, val_l1_hist, "-s", label="val hole-L1")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("Loss")
    ax[1].plot(epoch_hist, val_psnr_hist, "-^", color="green")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("PSNR (dB)"); ax[1].set_title("Validation PSNR")
    for a in ax: a.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(EVAL_DIR, "loss_curve.png"), dpi=120)
    display(fig); plt.close(fig)

target_epoch = completed_epochs + EPOCHS_TO_RUN
for _ in range(EPOCHS_TO_RUN):
    completed_epochs += 1
    epoch = completed_epochs

    net_E.train(); net_G.train()
    running, n_batches = 0.0, 0
    for imgs in train_loader:
        masks = get_large_random_mask(imgs.size(0), 128, 128, torch.device("cpu"))  # 1=hole
        comp = core.train_step(net_E, net_G, optimizer, imgs, masks,
                               LAMBDA_KL, LAMBDA_REC, OUTPUT_SCALE, DEVICE)
        running += comp["total"]; n_batches += 1
    train_loss = running / max(n_batches, 1)

    # ---- (1) validation loss, updated every epoch ----
    val_l1, val_psnr = core.evaluate_set(net_E, net_G, val_images, val_masks, BATCH_SIZE, DEVICE)
    epoch_hist.append(epoch); train_loss_hist.append(train_loss)
    val_l1_hist.append(val_l1); val_psnr_hist.append(val_psnr)

    # ---- (2) checkpoints: per-epoch + latest + best + resumable train_state ----
    core.save_checkpoint(net_E, net_G, OUTPUT_DIR, f"epoch_{epoch}")
    core.save_checkpoint(net_E, net_G, OUTPUT_DIR, "latest")
    if val_l1 < best_val_l1:
        best_val_l1, best_epoch = val_l1, epoch
        core.save_checkpoint(net_E, net_G, OUTPUT_DIR, "best")
    core.save_train_state(
        os.path.join(OUTPUT_DIR, "train_state.pt"), optimizer, completed_epochs,
        {"epoch": epoch_hist, "train": train_loss_hist, "val_l1": val_l1_hist, "val_psnr": val_psnr_hist},
        best_val_l1, best_epoch,
    )

    plot_curves()
    print(f"epoch {epoch:3d}/{target_epoch}  train={train_loss:.4f}  "
          f"val_L1={val_l1:.4f}  val_PSNR={val_psnr:.2f}dB  "
          f"(best L1={best_val_l1:.4f} @ epoch {best_epoch})")

    # ---- (3) periodic inference on the fixed validation image ----
    if epoch % VIS_EVERY == 0 or epoch == target_epoch:
        show_completions(net_E, net_G, vis_image, vis_mask, SAMPLE_NUM,
                         title=f"Fine-tuned (epoch {epoch}) — fixed validation image")

print(f"done. completed {completed_epochs} epochs total; "
      f"best val hole-L1 = {best_val_l1:.4f} at epoch {best_epoch} -> best_net_*.pth")

## 9. Comparison — pretrained vs. fine-tuned

Loads the frozen pretrained model and our **best** fine-tuned weights, and runs both on the
same held-out validation images with the same masks.

In [ ]:
pre_E, pre_G = get_pic_inpainter(PRETRAINED_DIR, DEVICE)
ft_E, ft_G, en, gn = core.load_model(OUTPUT_DIR, DEVICE)   # best_ > latest_ > highest epoch_N
print(f"fine-tuned checkpoint: {en} / {gn}")

# A fixed batch of val images for the visual comparison.
ncmp = min(4, val_images.size(0))
cmp_images = val_images[:ncmp]
cmp_masks  = val_masks[:ncmp]

print("PRETRAINED (ImageNet):")
show_completions(pre_E, pre_G, cmp_images, cmp_masks, SAMPLE_NUM, title="Pretrained (ImageNet)")
print("FINE-TUNED (flowers, L1+KL):")
show_completions(ft_E,  ft_G,  cmp_images, cmp_masks, SAMPLE_NUM, title="Fine-tuned (flowers, L1+KL)")

### Quantitative comparison over the full validation set

In [ ]:
pre_l1, pre_psnr = core.evaluate_set(pre_E, pre_G, val_images, val_masks, BATCH_SIZE, DEVICE)
ft_l1,  ft_psnr  = core.evaluate_set(ft_E,  ft_G,  val_images, val_masks, BATCH_SIZE, DEVICE)

print(f"{'model':<24}{'hole-L1 (lower better)':>26}{'PSNR dB (higher better)':>26}")
print(f"{'pretrained (ImageNet)':<24}{pre_l1:>26.4f}{pre_psnr:>26.2f}")
print(f"{'fine-tuned (L1+KL)':<24}{ft_l1:>26.4f}{ft_psnr:>26.2f}")
print(f"{'improvement':<24}{pre_l1 - ft_l1:>26.4f}{ft_psnr - pre_psnr:>26.2f}")

## 10. Where the weights are

`checkpoints/finetuned/` holds `epoch_{n}_net_{E,G}.pth`, `latest_net_{E,G}.pth`,
`best_net_{E,G}.pth`, and a resumable `train_state.pt`.

Download `best_net_{E,G}.pth` from Colab into `pic_inpainting/checkpoints/finetuned/` in the
repo. The tooling loads whatever pair lives in that folder (`best_` > `latest_` > highest
`epoch_N`), so no renaming is needed:

```bash
python pic_finetuning/run_finetuned.py
python pic_inpainting/evaluate_pic.py --ckpt_dir pic_inpainting/checkpoints/finetuned
```